# 1. Data Cleaning

Lanjutan dari audit. Data mentah Januari sampai Juni 2026 dibersihkan di sini, hasilnya disimpan ke `datasets/interim/`.

Catatan:
- `datasets/raw` tidak disentuh, hasil cleaning ditulis ke file baru.
- Tiap aturan dijalankan berurutan, baris sebelum dan sesudah dicatat ke `03_cleaning_log.csv`.

# 2. Import Library

In [1]:
import pandas as pd
import numpy as np
import os

# 3. Muat Data Mentah 6 Bulan

In [2]:
df = {}

for i in range(1, 7):
    path = rf"..\datasets\raw\2026\yellow_tripdata_2026-{i:02d}.parquet"
    df[i] = pd.read_parquet(path)
    print(f"Bulan {i}: {len(df[i]):,} baris, {df[i].shape[1]} kolom")

Bulan 1: 3,724,889 baris, 20 kolom
Bulan 2: 3,399,866 baris, 20 kolom
Bulan 3: 3,952,451 baris, 20 kolom
Bulan 4: 3,831,240 baris, 20 kolom
Bulan 5: 4,090,836 baris, 20 kolom
Bulan 6: 3,837,248 baris, 21 kolom


## 3.1 Buang Kolom `request_source`

Kolom ini cuma ada di bulan Juni dan 73,6% isinya kosong, jadi dibuang biar skema 6 bulan sama (20 kolom).

In [3]:
for i in range(1, 7):
    if "request_source" in df[i].columns:
        n = df[i]["request_source"].notna().sum()
        df[i] = df[i].drop(columns=["request_source"])
        print(f"Bulan {i}: request_source dibuang (non-null: {n:,})")

kolom = list(df[1].columns)
print("Semua bulan sama kolom:", all(list(df[i].columns) == kolom for i in range(1, 7)))

Bulan 6: request_source dibuang (non-null: 1,013,180)
Semua bulan sama kolom: True


# 4. Gabung Semua Bulan

In [4]:
df_all = pd.concat(df.values(), ignore_index=True)

# simpan asal bulan tiap baris
df_all["_bulan"] = np.repeat([1, 2, 3, 4, 5, 6], [len(df[i]) for i in range(1, 7)])

n_awal = len(df_all)
print(f"Total baris: {n_awal:,}")

Total baris: 22,836,530


# 5. Cleaning (C1-C10)

Aturan C1-C10 diambil dari `docs/analysis_plan.md`, dijalankan satu per satu lalu dicatat ke tabel attrition.

`main_trips` dipakai sebagai sumber utama analisis demand. Aturan nilai transaksi (jarak, tarif, penumpang) tetap diterapkan biar file ini sekalian layak untuk analisis revenue.

In [5]:
attrition = []

# helper buat nyatet tiap aturan ke tabel attrition
def catat(rule, n_before, alasan):
    n_buang = n_before - len(df_all)
    attrition.append({
        "Aturan": rule,
        "Baris sebelum": n_before,
        "Baris dibuang": n_buang,
        "Baris sesudah": len(df_all),
        "% dibuang": round(n_buang / n_before * 100, 4),
        "Alasan": alasan,
    })
    print(f"{rule}: dibuang {n_buang:,}, sisa {len(df_all):,}")

print(f"Baris awal: {len(df_all):,}")

Baris awal: 22,836,530


## C1 - Pickup di luar Januari-Juni 2026

In [6]:
n_sebelum = len(df_all)
mask = (df_all["tpep_pickup_datetime"] < "2026-01-01") | (df_all["tpep_pickup_datetime"] > "2026-06-30 23:59:59")
df_all = df_all[~mask].copy()
catat("C1 - Pickup di luar Jan-Jun 2026", n_sebelum, "Pickup di luar periode Januari-Juni 2026")

MemoryError: Unable to allocate 2.21 GiB for an array with shape (13, 22836515) and data type float64

## C2 - Timestamp kosong

In [ ]:
n_sebelum = len(df_all)
mask = df_all["tpep_pickup_datetime"].isna() | df_all["tpep_dropoff_datetime"].isna()
df_all = df_all[~mask].copy()
catat("C2 - Timestamp kosong", n_sebelum, "tpep_pickup_datetime atau tpep_dropoff_datetime kosong")

## C3 - Durasi tidak wajar

In [ ]:
n_sebelum = len(df_all)
durasi = (df_all["tpep_dropoff_datetime"] - df_all["tpep_pickup_datetime"]).dt.total_seconds() / 60
mask = (durasi <= 0) | (durasi > 24 * 60)  # dropoff sebelum pickup atau durasi lebih dari 24 jam
df_all = df_all[~mask].copy()
catat("C3 - Durasi tidak wajar", n_sebelum, "Dropoff sebelum pickup atau durasi lebih dari 24 jam")

## C4 - Zona pickup tidak valid

LocationID valid 1-263. ID 264 = Unknown, 265 = Outside of NYC, dua-duanya dibuang karena pickup wajib punya lokasi.

In [ ]:
valid_zone = set(range(1, 264))
n_sebelum = len(df_all)
mask = df_all["PULocationID"].isna() | (~df_all["PULocationID"].isin(valid_zone))
df_all = df_all[~mask].copy()
catat("C4 - Zona pickup tidak valid", n_sebelum, "PULocationID kosong, di luar 1-263, atau Unknown (264/265)")

## C5 - Zona dropoff tidak valid

In [ ]:
n_sebelum = len(df_all)
mask = df_all["DOLocationID"].isna() | (~df_all["DOLocationID"].isin(valid_zone))
df_all = df_all[~mask].copy()
catat("C5 - Zona dropoff tidak valid", n_sebelum, "DOLocationID kosong, di luar 1-263, atau Unknown (264/265)")

## C6 - Duplikat seluruh baris

In [ ]:
n_sebelum = len(df_all)
n_dup = int(df_all.duplicated().sum())
df_all = df_all.drop_duplicates().copy()
catat("C6 - Duplikat seluruh baris", n_sebelum, f"Seluruh baris identik ({n_dup:,} duplikat)")

## C7 - `trip_distance` tidak wajar

In [ ]:
n_sebelum = len(df_all)
mask = (df_all["trip_distance"] < 0) | (df_all["trip_distance"] > 100)  # nilai kosong dibiarkan
df_all = df_all[~mask].copy()
catat("C7 - trip_distance tidak wajar", n_sebelum, "Negatif atau lebih dari 100 mil")

## C8 - `fare_amount` / `total_amount` tidak wajar

In [ ]:
n_sebelum = len(df_all)
mask = (df_all["fare_amount"] < 0) | (df_all["fare_amount"] > 500)
mask = mask | (df_all["total_amount"] < 0) | (df_all["total_amount"] > 500)  # nilai kosong dibiarkan
df_all = df_all[~mask].copy()
catat("C8 - Tarif tidak wajar", n_sebelum, "fare_amount atau total_amount negatif atau lebih dari $500")

## C9 - `passenger_count` tidak wajar

In [ ]:
n_sebelum = len(df_all)
mask = (df_all["passenger_count"] < 0) | (df_all["passenger_count"] > 6)  # NaN dibiarkan, cuma nilai aneh yang dibuang
df_all = df_all[~mask].copy()
catat("C9 - Jumlah penumpang tidak wajar", n_sebelum, "passenger_count negatif atau lebih dari 6")

## C10 - Kode kategori tidak dikenal

In [ ]:
valid_vendor = {1, 2, 6, 7}
valid_ratecode = {1, 2, 3, 4, 5, 6, 99}
valid_payment = {0, 1, 2, 3, 4, 5, 6}

n_sebelum = len(df_all)
mask = (~df_all["VendorID"].isin(valid_vendor)) | (~df_all["payment_type"].isin(valid_payment))
# RatecodeID banyak yang kosong, jadi NaN dibiarkan; cuma kode asing yang dibuang
mask = mask | (df_all["RatecodeID"].notna() & ~df_all["RatecodeID"].isin(valid_ratecode))
df_all = df_all[~mask].copy()
catat("C10 - Kode kategori tidak dikenal", n_sebelum, "VendorID, RatecodeID, atau payment_type di luar daftar TLC")

## 5.1 Ringkasan Attrition

Tabel attrition disimpan ke `out/output_eda/03_cleaning_log.csv`.

In [ ]:
attrition_df = pd.DataFrame(attrition)
total_buang = int(attrition_df["Baris dibuang"].sum())
attrition_df.loc["Total"] = ["Total", n_awal, total_buang, len(df_all), round(total_buang / n_awal * 100, 4), ""]

display(attrition_df)

os.makedirs("../out/output_eda", exist_ok=True)
attrition_df.to_csv("../out/output_eda/03_cleaning_log.csv", index=False)

print(f"Total dibuang: {total_buang:,} ({round(total_buang / n_awal * 100, 2)}%)")
print(f"Baris akhir main_trips: {len(df_all):,}")

# 6. Ganti ID Lokasi Jadi Koordinat Centroid

Tabel `datasets/zone/taxi_zone.parquet` sudah punya `latitude`/`longitude` (centroid WGS84) per zona, jadi tinggal dipakai. Tabel pemetaan disimpan ke `datasets/zone/zone_centroid_lookup.csv`.

In [ ]:
zone = pd.read_parquet("../datasets/zone/taxi_zone.parquet")
zone = zone.rename(columns={"Zone": "zone_name", "Borough": "borough"})

nama = zone["zone_name"].str.lower()
zone["is_airport"] = ((zone["borough"].str.lower() == "ewr") | nama.str.contains("airport|newark|jfk|laguardia", regex=True)).astype(int)
zone["is_cbd"] = (zone["borough"].str.lower() == "manhattan").astype(int)

zone_centroid = zone[["LocationID", "longitude", "latitude", "zone_name", "borough", "is_airport", "is_cbd"]].copy()
zone_centroid.to_csv("../datasets/zone/zone_centroid_lookup.csv", index=False)

print("Jumlah zona:", len(zone_centroid))
print("Zona tanpa koordinat:", int(zone_centroid["longitude"].isna().sum()))
print("Satu zona satu record:", zone_centroid["LocationID"].is_unique)
display(zone_centroid.head(10))

## 6.1 Join koordinat ke `main_trips`

In [ ]:
lookup = zone_centroid.set_index("LocationID")
kolom_zona = ["longitude", "latitude", "zone_name", "borough", "is_airport", "is_cbd"]

df_all = df_all.reset_index(drop=True)  # index rapi biar concat tidak geser
n_sebelum = len(df_all)

pu = df_all[["PULocationID"]].join(lookup[kolom_zona], on="PULocationID").add_prefix("PU_").drop(columns=["PU_PULocationID"])
do = df_all[["DOLocationID"]].join(lookup[kolom_zona], on="DOLocationID").add_prefix("DO_").drop(columns=["DO_DOLocationID"])

df_main = pd.concat([df_all, pu, do], axis=1)

print(f"Baris sebelum join: {n_sebelum:,}, setelah join: {len(df_main):,}")
print("Koordinat pickup kosong:", int(df_main["PU_longitude"].isna().sum()))
print("Koordinat dropoff kosong:", int(df_main["DO_longitude"].isna().sum()))

## 6.2 Lepas kolom ID dan simpan `main_trips`

`datasets/raw` tetap utuh. Kolom ID cuma dilepas di salinan interim.

In [ ]:
df_main = df_main.drop(columns=["PULocationID", "DOLocationID"])

# rapikan urutan kolom
depan = ["tpep_pickup_datetime", "tpep_dropoff_datetime",
         "PU_longitude", "PU_latitude", "DO_longitude", "DO_latitude",
         "PU_zone_name", "PU_borough", "PU_is_airport", "PU_is_cbd",
         "DO_zone_name", "DO_borough", "DO_is_airport", "DO_is_cbd"]
sisa = [c for c in df_main.columns if c not in depan + ["_bulan"]]
df_main = df_main[depan + sisa + ["_bulan"]]

os.makedirs("../datasets/interim/2026", exist_ok=True)
df_main.to_parquet("../datasets/interim/2026/main_trips.parquet", index=False)

print("Shape main_trips:", df_main.shape)
display(df_main.head())

## 6.3 Cek ringkasan

In [ ]:
n_akhir = len(df_main)
print(f"{n_awal:,} -> {n_akhir:,} baris (dibuang {n_awal - n_akhir:,}, {round((n_awal - n_akhir) / n_awal * 100, 2)}%)")
print("Pasangan PU unik:", len(df_main[["PU_longitude", "PU_latitude"]].drop_duplicates()))